# Notebook 05 — Catálogo de Dados

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Documentar todas as tabelas do pipeline dentro do próprio Unity Catalog, descrevendo cada campo, seu tipo, seu domínio de valores e sua origem.

### Entrada e saída

| | |
|---|---|
| **Entrada** | As 7 tabelas das camadas bronze, silver e gold |
| **Saída** | Comentários gravados no Unity Catalog, visíveis no Catalog Explorer |

### Por que o catálogo vive dentro da plataforma

A alternativa seria escrever a documentação num arquivo separado. O problema é conhecido: documento à parte desatualiza. Alguém altera uma coluna, esquece de atualizar o Word, e a documentação passa a mentir — o que é pior que não ter documentação.

Gravando as descrições no Unity Catalog com `COMMENT ON TABLE` e `ALTER COLUMN ... COMMENT`, a documentação fica **acoplada ao dado**. Quem abre a tabela no Catalog Explorer vê a descrição de cada campo ali mesmo, e o Databricks ainda gera a linhagem automaticamente a partir das execuções.

Sem catálogo, dados viram caixa-preta: ninguém sabe o que cada campo significa, quais valores são válidos, nem de onde vieram.

## 1. Perfil de dados

Antes de documentar os domínios, é preciso conhecê-los. Esta consulta levanta mínimo, máximo, média e contagem de nulos de cada coluna numérica da camada Silver.

Serve a dois propósitos: confirmar que os valores observados são coerentes com as regras declaradas pelo ONS, e fornecer insumo factual para as descrições do catálogo.

In [0]:
from pyspark.sql import functions as F

perfil = spark.table("workspace.silver.balanco_energia")
cols = ["val_gerhidraulica","val_gertermica","val_gereolica",
        "val_gersolar","val_carga","val_intercambio"]

display(perfil.select([
    F.struct(
        F.lit(c).alias("coluna"),
        F.min(c).alias("minimo"),
        F.max(c).alias("maximo"),
        F.round(F.avg(c), 2).alias("media"),
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias("nulos")
    ).alias(c) for c in cols
]))

val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
"List(val_gerhidraulica, 136.934, 87875.71, 18897.05, 0)","List(val_gertermica, 19.0, 22751.102, 4259.62, 0)","List(val_gereolica, 0.0, 25064.762, 3776.94, 0)","List(val_gersolar, 0.0, 42551.688, 1546.89, 0)","List(val_carga, 1364.251, 106148.66, 28432.56, 0)","List(val_intercambio, -16172.03099999, 14146.508, 48.04, 0)"


## 2. Gravação do catálogo no Unity Catalog

O catálogo é definido como um dicionário Python e aplicado por um laço, em vez de dezenas de comandos SQL escritos à mão. A definição fica legível e centralizada, e acrescentar uma tabela é acrescentar uma entrada.

Cada descrição de coluna segue o mesmo padrão, cobrindo o que o enunciado exige de um catálogo de dados:

| Elemento | Exemplo |
|---|---|
| **O que o campo representa** | "Intercâmbio líquido de energia" |
| **Unidade** | "em MWmed" |
| **Domínio de valores** | "Único campo que admite valores negativos: negativo indica importação, positivo indica exportação" |
| **Origem e transformação** | "Coluna homônima do CSV do ONS" |

Repare que os comentários da camada Bronze dizem explicitamente "texto bruto" e explicam *por que* o campo está como texto. Isso não é redundância: é o catálogo registrando uma **decisão de arquitetura**. Quem abrir a tabela daqui a um ano vai entender que o tipo string foi intencional, não descuido.

Nem todas as tabelas do projeto são documentadas aqui. A `qa_violacoes_balanco`
nasce no notebook 06 e as três tabelas da extensão nascem no notebook 07 — cada
uma é documentada no notebook que a cria. Concentrar tudo neste ponto quebraria
uma execução limpa do pipeline, porque os comandos tentariam comentar tabelas que
ainda não existem. A cobertura total é verificada ao final do notebook 07.

In [0]:
catalogo = {
 "workspace.bronze.balanco_energia_bruto": {
   "_tabela": "Camada Bronze. Dados brutos do Balanco de Energia nos Subsistemas (ONS), "
              "preservados como texto exatamente como recebidos. Fonte: dados.ons.org.br, licenca CC-BY. "
              "Granularidade: hora x subsistema. Periodo: 2019-2025.",
   "id_subsistema":     "Texto bruto. Codigo do subsistema conforme recebido do CSV. Dominio: NE, N, S, SE, SIN.",
   "nom_subsistema":    "Texto bruto. Nome do subsistema por extenso conforme recebido.",
   "din_instante":      "Texto bruto. Data e hora no formato aaaa-MM-dd HH:mm:ss, horario de Brasilia. Convertido para timestamp apenas na camada Silver.",
   "val_gerhidraulica": "Texto bruto. Geracao hidraulica em MWmed. Mantido como texto para preservar a representacao original, incluindo notacao cientifica.",
   "val_gertermica":    "Texto bruto. Geracao termica em MWmed.",
   "val_gereolica":     "Texto bruto. Geracao eolica em MWmed.",
   "val_gersolar":      "Texto bruto. Geracao fotovoltaica em MWmed. Zeros aparecem na origem como 0E-8.",
   "val_carga":         "Texto bruto. Carga de energia verificada em MWmed.",
   "val_intercambio":   "Texto bruto. Intercambio liquido em MWmed. Admite valores negativos.",
   "_arquivo_origem":   "Metadado de linhagem. Nome do arquivo CSV que originou o registro.",
   "_data_ingestao":    "Metadado de linhagem. Momento da carga na camada Bronze.",
 },
 "workspace.silver.balanco_energia": {
   "_tabela": "Camada Silver. Dados tipados e padronizados. Origem: bronze.balanco_energia_bruto. "
              "Transformacoes: cast de texto para timestamp e double, trim de texto, "
              "marcacao da linha agregada SIN. Nenhuma linha foi removida.",
   "id_subsistema":  "Codigo do subsistema. Dominio: NE, N, S, SE, SIN. Origem: coluna homonima do CSV do ONS.",
   "nom_subsistema": "Nome do subsistema por extenso. Dominio: NORDESTE, NORTE, SUL, SUDESTE/CENTRO-OESTE, SISTEMA INTERLIGADO NACIONAL.",
   "din_instante":   "Data e hora da medicao, em horario de Brasilia conforme publicado pelo ONS. Granularidade horaria.",
   "val_gerhidraulica": "Geracao hidraulica verificada, em MWmed. Dominio: maior ou igual a zero. Aceita nulo e zero, nunca negativo.",
   "val_gertermica":    "Geracao termica verificada, em MWmed. Dominio: maior ou igual a zero.",
   "val_gereolica":     "Geracao eolica verificada, em MWmed. Dominio: maior ou igual a zero.",
    "val_gersolar":      "Geracao fotovoltaica verificada, em MWmed. Dominio: maior ou igual a zero. Zeros podem aparecer na origem em notacao cientifica (0E-8), convertidos para 0.0 no cast.",
   "val_carga":         "Carga de energia verificada, em MWmed. Dominio: maior ou igual a zero, nao aceita nulo.",
   "val_intercambio":   "Intercambio liquido de energia, em MWmed. Unico campo que admite valores negativos: negativo indica importacao, positivo indica exportacao.",
   "flag_agregado":     "Verdadeiro quando a linha e o total nacional (SIN), que e a soma dos quatro subsistemas. Usado para evitar dupla contagem.",
   "_arquivo_origem":   "Nome do arquivo CSV de origem. Metadado de linhagem gerado na ingestao.",
   "_data_ingestao":    "Momento em que o registro foi carregado na camada Bronze. Metadado de linhagem.",
 },
 "workspace.gold.dim_subsistema": {
   "_tabela": "Dimensao de subsistemas do Sistema Interligado Nacional. Grao: um registro por subsistema.",
   "sk_subsistema":  "Chave substituta (surrogate key) do subsistema. Inteiro sequencial.",
   "id_subsistema":  "Chave natural do subsistema. Dominio: NE, N, S, SE, SIN.",
   "nom_subsistema": "Nome do subsistema por extenso.",
   "flag_agregado":  "Verdadeiro apenas para o SIN. As tabelas fato excluem registros com este indicador.",
 },
 "workspace.gold.dim_fonte": {
   "_tabela": "Dimensao de fontes de geracao. Grao: um registro por fonte. Construida a partir de conhecimento de dominio do setor eletrico.",
   "sk_fonte":          "Chave substituta da fonte de geracao.",
   "nom_fonte":         "Nome da fonte. Dominio: Hidraulica, Termica, Eolica, Fotovoltaica.",
   "col_origem":        "Coluna da camada Silver que deu origem a esta fonte apos o unpivot. Metadado de linhagem.",
   "flag_renovavel":    "Verdadeiro para fontes renovaveis: hidraulica, eolica e fotovoltaica.",
   "flag_intermitente": "Verdadeiro para fontes de geracao intermitente, dependentes de condicoes naturais instantaneas: eolica e fotovoltaica.",
 },
 "workspace.gold.dim_tempo": {
   "_tabela": "Dimensao temporal em granularidade horaria, de 2019 a 2025. Grao: um registro por hora.",
   "din_instante":    "Instante da medicao em granularidade horaria, chave natural da dimensao. Origem da chave substituta sk_tempo e da juncao com a camada Silver.",
   "sk_tempo":        "Chave substituta no formato aaaaMMddHH. Exemplo: 2019010100 representa 1 de janeiro de 2019, hora 00.",
   "data":            "Data da medicao, sem o componente de hora.",
   "ano":             "Ano. Dominio: 2019 a 2025.",
   "mes":             "Mes. Dominio: 1 a 12.",
   "dia":             "Dia do mes. Dominio: 1 a 31.",
   "hora":            "Hora do dia. Dominio: 0 a 23.",
   "trimestre":       "Trimestre do ano. Dominio: 1 a 4.",
   "dia_semana":      "Dia da semana. Dominio: 1 (domingo) a 7 (sabado).",
   "flag_fim_semana": "Verdadeiro para sabado e domingo.",
   "estacao":         "Estacao do ano no hemisferio sul. Dominio: Verao, Outono, Inverno, Primavera.",
 },
 "workspace.gold.fato_geracao": {
   "_tabela": "Tabela fato de geracao de energia. Grao: hora x subsistema x fonte. "
              "Exclui a linha agregada SIN para evitar dupla contagem. "
              "Obtida por unpivot das quatro colunas de geracao da camada Silver.",
   "sk_tempo":          "Chave estrangeira para dim_tempo.",
   "sk_subsistema":     "Chave estrangeira para dim_subsistema.",
   "sk_fonte":          "Chave estrangeira para dim_fonte.",
   "val_geracao_mwmed": "Geracao verificada no periodo, em MWmed. Dominio: maior ou igual a zero.",
 },
 "workspace.gold.fato_carga": {
   "_tabela": "Tabela fato de carga e intercambio. Grao: hora x subsistema. Exclui a linha agregada SIN.",
   "sk_tempo":              "Chave estrangeira para dim_tempo.",
   "sk_subsistema":         "Chave estrangeira para dim_subsistema.",
   "val_carga_mwmed":       "Carga de energia verificada, em MWmed. Dominio: maior ou igual a zero.",
   "val_intercambio_mwmed": "Intercambio liquido, em MWmed. Negativo indica importacao, positivo indica exportacao.",
 },
}

for tabela, campos in catalogo.items():
    for campo, texto in campos.items():
        if campo == "_tabela":
            spark.sql(f"COMMENT ON TABLE {tabela} IS '{texto}'")
        else:
            spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {campo} COMMENT '{texto}'")
    print(f"documentado: {tabela} ({len(campos)-1} colunas)")

documentado: workspace.bronze.balanco_energia_bruto (11 colunas)
documentado: workspace.silver.balanco_energia (12 colunas)
documentado: workspace.gold.dim_subsistema (4 colunas)
documentado: workspace.gold.dim_fonte (5 colunas)
documentado: workspace.gold.dim_tempo (11 colunas)
documentado: workspace.gold.fato_geracao (4 colunas)
documentado: workspace.gold.fato_carga (4 colunas)
